# Py Lab 6 — Reproducible pipeline
**PUBHLT 0411 · Python for Public Health Data Analysis**

Name: 

Submit this notebook as a `.ipynb` file.

---

Each exercise states the task and names the variables to create. Write your code in the empty cell below it and run the cell.

Use the exact variable names given — later exercises build on them.

## Before you start — the data

This lab reads:

- `pa_counties.csv`
- `pa_firearm_county_year.csv`

These live in a `data/` folder next to this notebook. Run the cell below once to confirm Python can see them before you begin.

In [ ]:
# Setup — run this cell once, before anything else.
#
# The lab's CSVs live in a `data/` folder next to this notebook. This cell
# does not download anything; it just checks Python can find them.
import os

FILES = [
    "pa_counties.csv",
    "pa_firearm_county_year.csv"
]

missing = [f for f in FILES if not os.path.exists(os.path.join("data", f))]

if missing:
    print("MISSING from data/:", ", ".join(missing))
    print("Working directory:", os.getcwd())
    print("Start Jupyter from the folder that holds data/, then re-run this cell.")
else:
    print("Data ready:", ", ".join(FILES))

## Load the inputs

Every exercise below works from these two files. Run this cell once.

In [ ]:
# The pipeline's inputs. Run this before the exercises.
import pandas as pd
import matplotlib.pyplot as plt

firearm = pd.read_csv("data/pa_firearm_county_year.csv")
counties = pd.read_csv("data/pa_counties.csv")

print(firearm.shape, counties.shape)

---

## Exercise 1 — Import and inspect

Load `data/pa_counties.csv` into `county_reference`. Print its shape, how many
counties fall into each `metro` category, and confirm the file has one row per county.

**Create:** `county_reference`

---

## Exercise 2 — Filter to one year

Build `f2023`, the 2023 rows only. Print its shape, how many `firearm_deaths`
values are missing, and what fraction of counties that leaves reporting.

**Create:** `f2023`, `n_missing`, `n_reporting`

---

## Exercise 3 — Clean, and decide about the blanks

Build `reporting`: the 2023 counties that report a `firearm_deaths` total,
dropping on **that column only**. Then show what the careless version costs, by naming the
counties it would discard even though their death total is known.

**Create:** `reporting`, `lost`

---

## Exercise 4 — Join

Merge `reporting` with the `county`, `metro`, and `population_2023` columns of
`county_reference`, matching **on county**. Confirm the row count is unchanged and that
nothing failed to match.

**Create:** `before`, `joined`

---

## Exercise 5 — Group and summarise

**Your turn — no scaffold.** Answer this question from `joined`:

> **Do Pennsylvanians living outside metropolitan counties die by firearm at a higher rate
> than those living in them?**

Produce a table called `summary`, one row per metro group, reporting for each group: **how
many counties it contains, its total firearm deaths, its total population, and its firearm
death rate per 100,000 people.** Print it.

**Create:** `summary`

---

## Exercise 6 — The error that raises nothing

Repair it, then measure the damage the broken version did: how many people
were sitting in the nonmetro denominator with no deaths above them, and how far that pushed
the nonmetro rate.

**Create:** `repaired`, `fixed`, `phantom`, `understated_by`

*The code this exercise refers to:*

In [ ]:
broken = f2023.merge(county_reference[["county", "metro", "population_2023"]],
                     on="county", how="left")
broken_summary = broken.groupby("metro").agg(
    deaths=("firearm_deaths", "sum"),
    population=("population_2023", "sum"),
)
broken_summary["rate_per_100k"] = (broken_summary["deaths"]
                                   / broken_summary["population"] * 100000)
print(broken_summary.round(1))

---

## Exercise 7 — Visualise

**Your turn — no scaffold.** A reader accepts that the nonmetro rate is higher and asks the
obvious follow-up: **what kind of firearm death is driving it?**

Build a table called `share` giving, for each metro group, its **total firearm deaths, its
total firearm suicides, and suicides as a percentage of firearm deaths.** Treat a suppressed
suicide count as zero. Then draw a figure that lets a reader compare the two percentages, and
print the table.


> **Requirements the figure must meet**  
> A labelled vertical axis · a title · **a baseline at zero** · each bar labelled with its
> value · colours drawn from the course palette (`#003594`, `#8a6400`).

**Create:** `share`, `bars`

---

## Exercise 8 — Describe the figure

**Your turn — no scaffold.** Someone using a screen reader will hear your description instead
of seeing the figure you drew in Exercise 7. Write it into a string called
`share_description`, and print it.

They must be able to reconstruct the figure from your words alone: **what kind of chart it
is, what the two groups are, what quantity is being shown, both of its values, and where the
vertical axis begins.** Describe what is on the page, not what it means — the conclusion is
the reader's to draw.

**Create:** `share_description`

---

## Exercise 9 — Report the finding

**Your turn — no scaffold.** Write the paragraph a reader would see, into a string called
`finding`, and print it.

It must state, **in prose**: how many counties reported a count and out of how many; both
pooled rates, identified by group; how many deaths the nonmetropolitan estimate rests on and
across how many counties; and that counties reporting no total were excluded from both the
numerator and the denominator.


> **Every number must be computed, not typed**  
> Build the sentence with an f-string reading from `summary` and `reporting`. **A number typed
> by hand stops being a result the moment the data changes.**

**Create:** `n_reporting`, `n_total`, `nonmetro_rate`, `metro_rate`, `nonmetro_deaths`, `nonmetro_counties`, `finding`